In [0]:
df_input = spark.read.csv("dbfs:/Volumes/7_outgoing/pharos/upload/pharos_pilot_list.csv", header=True, inferSchema=True)
display(df_input)

In [0]:
from pyspark.sql.functions import when, col

df_mp = spark.table("4_prod.raw.mill_person")
df_pid = df_input.join(df_mp, df_input.research_id == df_mp.PERSON_ID, "left")
df_pid = df_pid\
    .withColumn("matched", when(col("PERSON_ID").isNotNull(), True).otherwise(False))\
    .select(col("research_id").alias("person_id"), "matched")\
    .distinct()
display(df_pid)

In [0]:
from pyspark.sql.functions import col

# Load tables
df_examcode = spark.table("4_prod.pacs_dlt.pacs_examcode_dict")
df_imaging = spark.table("4_prod.pacs.imaging_metadata")


# Prepare ec DataFrame
df_ec = df_examcode.filter(
    (
        (col("preferred").ilike("MRI breast%")) & (col("preferred") != "MRI Breast implant Both")
    ) |
    (col("preferred").ilike("XR Mammogram%")) |
    (col("preferred").ilike("US Breast%")) |
    (col("preferred").ilike("US Axilla%"))
).select("short_code").distinct()

# Join all together
df_acn = df_imaging.join(df_pid, df_imaging.PersonId == df_pid.person_id, "inner") \
    .join(df_ec, df_imaging.ExamCode == df_ec.short_code, "inner")

df_acn = df_acn.select(col("AccessionNbr").alias("Accession_Number"))

# Save as table
df_acn.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("7_outgoing.pharos.pacs_accession_number_20260619")

In [0]:
display(df_acn)

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.pharos_20260522.imaging_report AS
SELECT DISTINCT ir.PersonId, ir.ReportEventId, ir.AccessionNbr, bd.AnonymizedText FROM 4_prod.pacs.imaging_report AS ir
LEFT JOIN 4_prod.rde.rde_blobdataset AS bd
ON ir.reporteventid = bd.eventid
INNER JOIN (SELECT DISTINCT AccessionNbr FROM 5_projects.pharos_20260522.extract_imaging) AS an
ON an.AccessionNbr = ir.AccessionNbr

In [0]:
%sql
SELECT COUNT(DISTINCT PersonId), COUNT(ReportEventId), COUNT(DISTINCT ReportEventId) FROM 5_projects.pharos_20260522.imaging_report
LIMIT 300

In [0]:
dbutils.fs.cp(
       "dbfs:/Volumes/1_inland/sectra/vone/pharos_20260522/",
       "dbfs:/Volumes/7_outgoing/pharos/imaging/pharos_20260522/",
       recurse=True
   )